# SAADE v5 · 01 — Справжня генерація Text → TTS та перевірка походження

Синтезуємо той самий **перевірений текст**, який звучить в оригінальному REAL-записі. Згенеровані файли й SHA-256 записуються тільки в `audio_real_fake/results/SAADE_AUA_2026_v5_FINAL/paired_tts_generated`. За відсутності TTS-моделі код не імітує мовлення.

Цей ноутбук запускається **після `00_Preflight_AUA.ipynb`**. Кожен застосований TTS-голос належить лише одному поділу; для коректного групового bootstrap потрібні незалежні голоси в adaptation/validation/hidden test. Частина **train**-голосів буде перенесена алгоритмом до adaptation після створення пар.

Перед генерацією обов'язково перевіряються транскрипції та умови використання аудіо й текстів. Підтвердження не виконується автоматично.

## 0. Налаштування

In [ ]:
from pathlib import Path
from dataclasses import replace
import sys, json, shutil, importlib.util
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT = Path('/home/jovyan/work/audio_real_fake')
AUA = PROJECT / 'AUA'
RESULTS = PROJECT / 'results'
OUT = RESULTS / 'SAADE_AUA_2026_v5_FINAL'
assert AUA.is_dir(), f'Не знайдено AUA: {AUA}'
assert shutil.which('ffmpeg'), 'Потрібно встановити ffmpeg, щоб відтворити MP3-сценарії'
sys.path.insert(0,str(Path.cwd().resolve()))
import saade_final_core as S
import saade_final_data as D
import saade_final_methods as M
import saade_final_visuals as V
import saade_final_adaptive as A
for mod in (S,D,M,V,A):
    assert Path(mod.__file__).resolve().parent == Path.cwd().resolve(), f'Чужа версія модуля: {mod.__file__}'
cfg=S.Config(root=str(AUA),out=str(OUT),scenario_profile='v2_5level',
             seed=42,seconds=4.,min_seconds=2.,augmentation_probability=.50,
             batch_size=12,lr=.0005,epochs=15,use_cache=True)
print('AUA:',AUA,'\nRESULTS:',OUT)
print('Дані лише для читання. Melnyk_dus не використовується.')

## 1. Перевірка черги та розподілу текстів

In [ ]:
QUEUE_FILE=OUT/'V5_00_TTS_GENERATION_QUEUE.csv'
assert QUEUE_FILE.is_file(),'Спершу виконайте 00_Preflight_AUA.ipynb'
queue=pd.read_csv(QUEUE_FILE,dtype=str,keep_default_na=False)
display(queue.groupby(['split','tts_status']).size().rename('n').reset_index())
display(queue[['pair_id','relative_path','split','text_normalized','source_license',
               'source_approval_status','tts_status','tts_voice_id']].head(12))
print('Рядки:',len(queue))

## 2. Призначення голосів і перевірка текстів (ручне рішення)

Для запуску локального **Piper** потрібні реальні україномовні `.onnx` і відповідні `.onnx.json` поруч з файлами. Укажіть файли голосів у словнику, а в CSV позначте перевірені записи `tts_status=APPROVED` та заповніть `tts_voice_id` і `tts_model_id`.

**Не використовуйте той самий голос у різних split.** Для незалежного тесту плануйте щонайменше 3 голоси у validation і hidden test та достатньо різних голосів у train, щоб із них виділити щонайменше 3 групи adaptation. Вихідна TTS-черга ще не має adaptation. Створювати голоси / параметри лише перейменуванням одного ONNX не можна.

In [ ]:
# Приклад конфігурації: замініть своїми реальними файлами .onnx.
# У словнику ключ має відповідати tts_voice_id у затвердженому CSV.
PIPER_MODELS_BY_VOICE = {
    # 'train_voice_01': '/home/jovyan/work/audio_real_fake/tts_models/uk_01.onnx',
    # 'train_voice_02': '/home/jovyan/work/audio_real_fake/tts_models/uk_02.onnx',
    # 'validation_voice_01': '/home/jovyan/work/audio_real_fake/tts_models/uk_03.onnx',
    # 'hidden_voice_01': '/home/jovyan/work/audio_real_fake/tts_models/uk_04.onnx',
}

# Щоб убезпечити дані, прапорець генерації спочатку вимкнений.
GENERATE_REVIEWED_AUDIO = False
approved=queue[queue.tts_status.eq('APPROVED')].copy()
print('Підтверджених завдань:',len(approved))
if len(approved):
    assert approved.tts_voice_id.str.strip().ne('').all()
    assert approved.tts_model_id.str.strip().ne('').all()
    voices=(approved.assign(key=approved.tts_voice_id.str.casefold().str.strip())
            .groupby('key').split.nunique())
    assert not voices.gt(1).any(), 'Голос використано в кількох split'
if GENERATE_REVIEWED_AUDIO:
    assert len(approved)>0, 'Немає схвалених завдань'
    receipts=A.synthesize_piper_reviewed(QUEUE_FILE,cfg,PIPER_MODELS_BY_VOICE)
    display(receipts.head())
else:
    print('Генерація ще не запущена. Перевірте тексти/ліцензії, підготуйте Piper та встановіть GENERATE_REVIEWED_AUDIO=True.')

## 3. Перевірка згенерованих файлів

CSV receipts — єдине джерело доказів фактичного Text→TTS, у ньому SHA-256, реальний модельний голос, шлях, оригінальний запис, текст і split.

In [ ]:
RECEIPTS=OUT/'V5_00_PIPER_GENERATION_RECEIPTS.csv'
if RECEIPTS.is_file():
    r=pd.read_csv(RECEIPTS,dtype=str,keep_default_na=False)
    display(r.groupby(['split','model_id','voice_id']).size().rename('audio_files').reset_index())
    print('Згенеровано:',len(r),'files')
else:
    print('Ще немає реальних TTS-файлів — matched_tts не можна запускати.')

## 4. Наступний етап

Коли `V5_00_PIPER_GENERATION_RECEIPTS.csv` заповнений справжніми аудіофайлами і є достатньо незалежних голосів, у головному ноутбуці встановіть `DATA_MODE="matched_tts"`. Інакше запуск `pilot_existing` — лише пілот, не дисертаційний доказ.